---
title: "2.7 Wide and long data; panel structure"
---

::: {.lesson-tools}
[Open in Colab](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/lessons/02-data/07-reshaping.ipynb){.btn .btn-primary}
[Download notebook](../../downloads/02-data/07-reshaping.ipynb){.btn .btn-outline-secondary}
:::

**Run in your browser.** Colab needs a Google account to run and save work. If its GitHub link cannot open, download this notebook and use **File → Upload notebook** at [Colab](https://colab.research.google.com/). You can read the website without an account. A notebook is a document containing explanatory text and runnable code cells.

**Time:** about 40 minutes, including exercises · **Before you begin:** Complete [2.6 Combining datasets safely](../02-data/06-combining.ipynb) first. Each notebook starts afresh: it does not remember variables from another lesson.

## You will learn

- Recognize wide and long representations of the same observations.
- Use pivot and melt without silently aggregating duplicates.
- Check unique country-year keys and a round trip.

## Why it matters

A **long** panel has repeated rows for each country, with a year column identifying time. A **wide** table places different years in separate columns. They can represent the same measurements, but some tasks need one shape or the other. Reshaping rearranges values; it should not invent observations or average conflicting rows without your decision. Begin with made-up numbers and verify the result by eye.

## Worked example

### Recognize a long panel

These fictional life expectancies form four country-year observations. The country and year together identify each measurement. `.duplicated([keys])` tests whether a key pair has already appeared.

In [ ]:
import pandas as pd
small_long = pd.DataFrame({
    "country": ["Northland", "Northland", "Eastria", "Eastria"],
    "year": [2002, 2007, 2002, 2007],
    "life_exp": [70.0, 74.0, 76.0, 78.0],
})
print("Repeated country-years:", small_long.duplicated(["country", "year"]).sum())
small_long

No country-year pair repeats. Repeated country names are expected in a panel; repeated country-year keys are a different issue. The unit of observation is the country-year.

### Pivot into a wide display

`.pivot` uses unique pairs to rearrange a table. `index="country"` identifies the rows, `columns="year"` the new columns and `values="life_exp"` the measurement. It does not average anything.

In [ ]:
small_wide = small_long.pivot(index="country", columns="year", values="life_exp")
small_wide

There are now two country rows and two year columns. Northland still has 70.0 in 2002 and 74.0 in 2007. A country appears once, but the table contains multiple measurements per row.

### Melt back to long form

`.reset_index()` turns the country row labels into a normal column. `.melt` gathers the year columns into rows. `id_vars` names fields that identify the rows; `var_name` names the gathered column labels, and `value_name` names their measurements. Convert the gathered years to integers for a clear numeric key.

In [ ]:
small_back = small_wide.reset_index().melt(id_vars="country", var_name="year", value_name="life_exp")
small_back["year"] = small_back["year"].astype(int)
small_back = small_back.sort_values(["country", "year"]).reset_index(drop=True)
small_back

The same four country-year values return, though their row order may differ. `.reset_index(drop=True)` discards old row labels after sorting. Compare keys and values rather than demanding the same incidental display order.

### Prepare the real country panel

Run this prepared setup once in a fresh session. It uses the course CSV in a local checkout, or downloads its fixed, checksum-verified public source in Colab. A checksum is a fingerprint of a file's contents; a mismatch stops the download from being used. You do not need to retype the setup. It does not use your Google Drive or any private data.

In [ ]:
# Prepared setup: use the course CSV locally; fetch its pinned source in Colab.
from pathlib import Path
import hashlib
import io
from urllib.request import urlopen
import pandas as pd

data_file = Path("data/gapminder.csv")
if not data_file.exists():
    source = "https://raw.githubusercontent.com/jennybc/gapminder/391b5a40d3574c3b89d889dd7d72f7f3fdb4a70c/inst/extdata/gapminder.tsv"
    with urlopen(source, timeout=30) as response:
        raw = response.read()
    expected = "f74275b4a2f4a7c8e3a09a752f9d6e43d1759d78d5cbfdebaaa3e1a6efc68f78"
    if hashlib.sha256(raw).hexdigest() != expected:
        raise ValueError("The data download changed. Please report this to the course maintainer.")
    downloaded = pd.read_csv(io.BytesIO(raw), sep="\t", dtype=str)
    downloaded = downloaded.rename(columns={"continent": "region", "lifeExp": "life_exp", "pop": "population", "gdpPercap": "gdp_pc"})
    data_file.parent.mkdir(parents=True, exist_ok=True)
    downloaded.to_csv(data_file, index=False, lineterminator="\n")
print("Course data ready: data/gapminder.csv")

Look for **Course data ready**. Colab working files are temporary; rerun this cell after a runtime reset. The data cover 142 countries at five-year intervals from 1952 to 2007. Each row is one country in one year.

### Reshape a real two-year comparison

Load the country panel and select 1952 and 2007. Keep only the fields relevant to this reshape. The full panel is already long, so this is a conversion of the same measurements, not a new data source.

In [ ]:
countries = pd.read_csv(data_file)
selected = countries.loc[countries["year"].isin([1952, 2007]), ["country", "year", "life_exp"]].copy()
print("Repeated keys:", selected.duplicated(["country", "year"]).sum())
wide = selected.pivot(index="country", columns="year", values="life_exp")
wide["change_years"] = wide[2007] - wide[1952]
wide.head()

The repeated-key count is zero. The new change column is a difference in **years of life expectancy** across 55 calendar years. It is not a percentage change or annual rate. Check integer column labels: after pivot, 2007 is a numeric label, not the string "2007".

### Select measurement columns before melting

The derived change column is not a third observation year. Select only the two actual years, then melt and restore the numeric year type.

In [ ]:
long_again = wide[[1952, 2007]].reset_index().melt(id_vars="country", var_name="year", value_name="life_exp")
long_again["year"] = long_again["year"].astype(int)
print(long_again.shape)
print("Missing life expectancies:", long_again["life_exp"].isna().sum())
long_again.head()

There are 284 observations and no missing life expectancies in this two-year teaching extract. Including change_years among the year columns would incorrectly turn a derived comparison into an observation. In an unbalanced panel, some countries are missing some dates; a wide display will show those gaps rather than create data.

### Inspect panel coverage

A **balanced panel** has the same observed periods for each unit. Count the number of distinct years for each country in the full panel, then count how many countries have each coverage count.

In [ ]:
years_per_country = countries.groupby("country")["year"].nunique()
print(years_per_country.value_counts().sort_index())

All 142 countries have 12 observed years in this filtered extract. Balanced coverage is convenient for teaching, but it does not establish representative sampling, measurement quality or the causal assumptions of a later panel model.

## Try it

Work in your own notebook copy. Add a new code cell for each task, try it yourself, then open the solution to compare. Run from the top if a name is missing.

### 1. Inspect one country

Display Canada's 1952 and 2007 life expectancies and the derived change from the wide table. State the units.

<details><summary>Solution 1</summary>

Run this complete solution after the worked example.

In [ ]:
wide.loc[["Canada"], [1952, 2007, "change_years"]]

Canada rises from 68.750 to 80.653 years in this extract, a gain of 11.903 years across the full period. This historical comparison does not identify the cause.

</details>

### 2. Reshape a different measurement

Use 1952 and 2007 GDP per person to create `income_wide`. Check its number of countries.

<details><summary>Solution 2</summary>

Run this complete solution after the worked example.

In [ ]:
income_selected = countries.loc[countries["year"].isin([1952, 2007]), ["country", "year", "gdp_pc"]]
income_wide = income_selected.pivot(index="country", columns="year", values="gdp_pc")
print(income_wide.shape)

The result has 142 country rows and two year columns. Its measurement is GDP per person, unlike the life-expectancy table with the same layout.

</details>

### 3. Check the returned values

Sort `selected` and `long_again` by country and year, reset their row labels, and compare the life-expectancy columns with `.equals`. Save the True/False result as `round_trip_matches`.

<details><summary>Solution 3</summary>

Run this complete solution after the worked example.

In [ ]:
original_sorted = selected.sort_values(["country", "year"]).reset_index(drop=True)
returned_sorted = long_again.sort_values(["country", "year"]).reset_index(drop=True)
round_trip_matches = original_sorted["life_exp"].equals(returned_sorted["life_exp"])
print(round_trip_matches)

The result is True. The build additionally checks country-year keys and numeric values, so an accidental reassignment to the wrong observation is not hidden by matching counts.

</details>

## When it goes wrong

The following examples deliberately trigger specific errors. The prepared `try`/`except` wrapper catches that named error and prints its message so the notebook can continue. It is for these demonstrations; an unexpected error elsewhere should still stop execution. You can read the message without memorizing the wrapper.

### Duplicate keys make a pivot ambiguous

Append a duplicate country-year and ask for an ordinary pivot. There would be two possible values for one output cell.

In [ ]:
try:
    duplicated_long = pd.concat([small_long, small_long.iloc[[0]]], ignore_index=True)
    duplicated_long.pivot(index="country", columns="year", values="life_exp")
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

Inspect duplicated country-year pairs before choosing a fix. A pivot_table with aggfunc="mean" would average them, which is an analytic decision, not merely a reshaping fix.

### Numeric and text column labels differ

The year columns came from integer year values.

In [ ]:
try:
    wide["2007"]
except KeyError as error:
    print(f"{type(error).__name__}: {error}")

Use the numeric column label 2007 here. Inspect wide.columns before guessing. A spreadsheet export may later convert headers to text, so check the imported labels each time.

## Recap

- Long and wide forms can contain the same measurements.
- Use unique keys with pivot; use melt to gather columns into rows.
- Exclude derived columns from the list of observation years.
- Compare keys and values after reshaping and inspect coverage.

**Next:** [2.8 Dates and time series basics](../02-data/08-dates-time-series.ipynb).

::: {.callout-note title="Coming from Stata / R / Excel?"}
This is the role of reshape wide/long in Stata and pivot_wider/pivot_longer in R. Excel-style visual layouts are often wide; modelling and grouped operations commonly use long data.
:::